In [ ]:
from google.colab import files
uploaded = files.upload()

Saving forecast_trail_1.csv to forecast_trail_1 (1).csv


In [ ]:

import pandas as pd
import hashlib

# 1) Load file
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# 2) Hashing function
def anonymize_email(email):
    if pd.isna(email):
        return None
    return hashlib.sha256(email.encode()).hexdigest()[:16]

# 3) Create anonymized ID
df["customer_id"] = df["customer_email"].apply(anonymize_email)

# 4) Drop original email
df = df.drop(columns=["customer_email"])

# 5) Check
df.head()


/tmp/ipython-input-2504631003.py:6: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


,customer_status,order_date,order_id,product_list,quantity_list,net_revenue,coupon_code,coupon_influencer,customer_id
0,NEW,2023-09-02,2000328999,Daily Gut Pulver-Strawberry,1,29.831776,achtsam,achtsam schlank,8cb4adeae32d800d
1,RETURNING,2023-09-02,2000329000,"Daily Gut Pulver-Apple,Gut Care Kapseln-NOFLAVOUR","1,1",59.663551,peachy,ameliandjana,99ba56718494f66f
2,RETURNING,2023-09-02,2000329008,"MCT C8 Pulver-NOFLAVOUR,Mood Kapseln-NOFLAVOUR...","1,1,1,1,1",112.152273,dsk,drsimonekoch,b714a5e095b2bc1b
3,RETURNING,2023-09-02,2000329025,Gut Care Kapseln-NOFLAVOUR,1,22.355140,spontent,spontent,1c183ebbec048c03
4,RETURNING,2023-09-02,2000329032,"Gut Care Kapseln-NOFLAVOUR,Gut Care Kapseln-NO...","1,3,1,1,2",128.245794,achtsam,achtsam schlank,40bb9082784dd165


In [ ]:
import pandas as pd

# 1) product_list'i split et
df["product_list"] = df["product_list"].astype(str).str.split(",")

# 2) quantity_list'i split et → integer'a çevir
df["quantity_list"] = df["quantity_list"].astype(str).str.split(",")
df["quantity_list"] = df["quantity_list"].apply(lambda lst: [int(x.strip()) for x in lst])

# 3) Ürün + Quantity birlikte explode
rows = []

for idx, row in df.iterrows():
    products = [p.strip() for p in row["product_list"]]
    quantities = row["quantity_list"]

    # Eğer uzunluklar eşit değilse → hata log
    if len(products) != len(quantities):
        print("⚠️ Quantity mismatch at order_id:", row["order_id"])
        continue

    for p, q in zip(products, quantities):
        rows.append({
            "customer_id": row["customer_id"],
            "customer_status": row["customer_status"],
            "order_date": row["order_date"],
            "order_id": row["order_id"],
            "product": p,
            "quantity": q,
            "net_revenue": row["net_revenue"],
            "coupon_code": row["coupon_code"],
            "coupon_influencer": row["coupon_influencer"]
        })

df_expanded = pd.DataFrame(rows)

df_expanded.head()


,customer_id,customer_status,order_date,order_id,product,quantity,net_revenue,coupon_code,coupon_influencer
0,8cb4adeae32d800d,NEW,2023-09-02,2000328999,Daily Gut Pulver-Strawberry,1,29.831776,achtsam,achtsam schlank
1,99ba56718494f66f,RETURNING,2023-09-02,2000329000,Daily Gut Pulver-Apple,1,59.663551,peachy,ameliandjana
2,99ba56718494f66f,RETURNING,2023-09-02,2000329000,Gut Care Kapseln-NOFLAVOUR,1,59.663551,peachy,ameliandjana
3,b714a5e095b2bc1b,RETURNING,2023-09-02,2000329008,MCT C8 Pulver-NOFLAVOUR,1,112.152273,dsk,drsimonekoch
4,b714a5e095b2bc1b,RETURNING,2023-09-02,2000329008,Mood Kapseln-NOFLAVOUR,1,112.152273,dsk,drsimonekoch


In [ ]:
import pandas as pd

sale_periods = [
    ("Winter_Sale_2024",  "2024-01-23", "2024-01-28"),
    ("Easter_Sale_2024",  "2024-03-28", "2024-04-03"),
    ("Summer_Sale_2024",  "2024-08-26", "2024-09-03"),
    ("Early_Bird_2024",   "2024-10-28", "2024-11-04"),
    ("Black_Week_2024",   "2024-11-25", "2024-12-02"),
    ("Winter_Sale_2025",  "2025-01-27", "2025-02-04"),
    ("Spring_Sale_2025",  "2025-04-23", "2025-05-01"),
    ("Summer_Sale_2025",  "2025-08-25", "2025-09-02"),
    ("Early_Bird_2025",   "2025-10-27", "2025-10-31"),
    ("Black_Week_2025",   "2025-11-24", "2025-12-01"),
]

# Tarih formatına çevir
sale_periods = [
    (name, pd.to_datetime(start), pd.to_datetime(end))
    for name, start, end in sale_periods
]


In [ ]:
def get_sale_name(date):
    for name, start, end in sale_periods:
        if start <= date <= end:
            return name
    return None


In [ ]:
df_expanded["order_date"] = pd.to_datetime(df_expanded["order_date"])

df_expanded["sale_name"] = df_expanded["order_date"].apply(get_sale_name)
df_expanded["is_sale_period"] = df_expanded["sale_name"].notna().astype(int)


In [ ]:
df_expanded.head()


,customer_id,customer_status,order_date,order_id,product,quantity,net_revenue,coupon_code,coupon_influencer,sale_name,is_sale_period
0,8cb4adeae32d800d,NEW,2023-09-02,2000328999,Daily Gut Pulver-Strawberry,1,29.831776,achtsam,achtsam schlank,None,0
1,99ba56718494f66f,RETURNING,2023-09-02,2000329000,Daily Gut Pulver-Apple,1,59.663551,peachy,ameliandjana,None,0
2,99ba56718494f66f,RETURNING,2023-09-02,2000329000,Gut Care Kapseln-NOFLAVOUR,1,59.663551,peachy,ameliandjana,None,0
3,b714a5e095b2bc1b,RETURNING,2023-09-02,2000329008,MCT C8 Pulver-NOFLAVOUR,1,112.152273,dsk,drsimonekoch,None,0
4,b714a5e095b2bc1b,RETURNING,2023-09-02,2000329008,Mood Kapseln-NOFLAVOUR,1,112.152273,dsk,drsimonekoch,None,0


In [ ]:
health_products = [
"Vitamin K2 Tropfen",
"Vitamin D3+K2 Tropfen",
"Vitamin D3 Omega Bundle",
"Vitamin D3 + Magnesium Bundle",
"Vitamin D + Test",
"Vitamin C & Zink Kapseln",
"Vitamin B12 Tropfen",
"Vitamin B12 Tabletten",
"Vitamin B Komplex",
"Vegan Omega-3 Kapseln",
"Vegan Omega 3",
"Zink Kapseln",
"Salbeiblatt Kapseln",
"Salbei Extrakt",
"Sägepalmen Kapseln",
"Sägepalmen Extrakt",
"Sägepalme Kapseln",
"Reishi Premium Kapseln",
"Reishi Forte Kapseln",
"Reishi Forte",
"Reishi Extrakt Kapseln",
"Premium Greens",
"Panax Ginseng Kapseln",
"Magnesium Komplex Kapseln",
"Panax Ginseng",
"Omega-3 Kapseln",
"L-Tyrosin",
"L-Tryptophan Kapseln",
"L-Lysin Kapseln",
"L-Glutathion Kapseln",
"Immune Protect Drink",
"Immune Gummies",
"Hydrate Drink",
"Hyaluronsäure Kapseln",
"Curcuma Kapseln",
"Curcuma Extrakt",
"Mood Kapseln",
"Curcuma & Piperin Plus",
"Vegan Basics",
"Minerals Kapseln"
"Colostrum Kapseln"
]


In [ ]:
df_expanded["product_clean"] = df_expanded["product"].str.split("-").str[0].str.strip()
df_expanded["is_health_product"] = df_expanded["product_clean"].isin(health_products).astype(int)


In [ ]:
health_affinity = (
    df_expanded.groupby("customer_id")["is_health_product"].mean()
    .rename("health_affinity")
)


In [ ]:
hydration_products = [
    "Hydrate Drink",
    "Recharge Drink",
    "Essential Aminos Drink",
]


In [ ]:
df_expanded["is_hydration_product"] = df_expanded["product_clean"].isin(hydration_products).astype(int)

hydration_affinity = (
    df_expanded.groupby("customer_id")["is_hydration_product"].mean()
    .rename("hydration_affinity")
)


In [ ]:
protein_products = [
    "Whey Protein",
    "Vegan Protein Pulver",
]


In [ ]:
df_expanded["is_protein_product"] = df_expanded["product_clean"].isin(protein_products).astype(int)

protein_affinity = (
    df_expanded.groupby("customer_id")["is_protein_product"].mean()
    .rename("protein_affinity")
)


In [ ]:
feature_affinities = pd.DataFrame({
    "health_affinity": health_affinity,
    "hydration_affinity": hydration_affinity,
    "protein_affinity": protein_affinity
}).reset_index()


In [ ]:
df_expanded.head()


,customer_id,customer_status,order_date,order_id,product,quantity,net_revenue,coupon_code,coupon_influencer,sale_name,is_sale_period,product_clean,is_health_product,is_hydration_product,is_protein_product
0,8cb4adeae32d800d,NEW,2023-09-02,2000328999,Daily Gut Pulver-Strawberry,1,29.831776,achtsam,achtsam schlank,None,0,Daily Gut Pulver,0,0,0
1,99ba56718494f66f,RETURNING,2023-09-02,2000329000,Daily Gut Pulver-Apple,1,59.663551,peachy,ameliandjana,None,0,Daily Gut Pulver,0,0,0
2,99ba56718494f66f,RETURNING,2023-09-02,2000329000,Gut Care Kapseln-NOFLAVOUR,1,59.663551,peachy,ameliandjana,None,0,Gut Care Kapseln,0,0,0
3,b714a5e095b2bc1b,RETURNING,2023-09-02,2000329008,MCT C8 Pulver-NOFLAVOUR,1,112.152273,dsk,drsimonekoch,None,0,MCT C8 Pulver,0,0,0
4,b714a5e095b2bc1b,RETURNING,2023-09-02,2000329008,Mood Kapseln-NOFLAVOUR,1,112.152273,dsk,drsimonekoch,None,0,Mood Kapseln,1,0,0


In [ ]:
df_expanded["is_prev_cocreation_product"] = (
    df_expanded["product"].str.contains("Vegan Protein Pulver", case=False) &
    df_expanded["product"].str.contains("Cookie", case=False)
).astype(int)


In [ ]:
prev_cocreation_affinity = (
    df_expanded.groupby("customer_id")["is_prev_cocreation_product"]
    .max()
    .rename("prev_cocreation_affinity")
)


In [ ]:
df_expanded["is_cocreation_influencer_customer"] = (
    df_expanded["coupon_influencer"].str.lower() == "fit_laura"
).astype(int)


In [ ]:
cocreation_influencer_affinity = (
    df_expanded.groupby("customer_id")["is_cocreation_influencer_customer"]
    .max()
    .rename("cocreation_influencer_affinity")
)


In [ ]:
feature_affinities = pd.DataFrame({
    "health_affinity": health_affinity,
    "hydration_affinity": hydration_affinity,
    "protein_affinity": protein_affinity,
    "prev_cocreation_affinity": prev_cocreation_affinity,
    "cocreation_influencer_affinity": cocreation_influencer_affinity
}).reset_index()
df_expanded.head()


,customer_id,customer_status,order_date,order_id,product,quantity,net_revenue,coupon_code,coupon_influencer,sale_name,is_sale_period,product_clean,is_health_product,is_hydration_product,is_protein_product,is_prev_cocreation_product,is_cocreation_influencer_customer
0,8cb4adeae32d800d,NEW,2023-09-02,2000328999,Daily Gut Pulver-Strawberry,1,29.831776,achtsam,achtsam schlank,None,0,Daily Gut Pulver,0,0,0,0,0
1,99ba56718494f66f,RETURNING,2023-09-02,2000329000,Daily Gut Pulver-Apple,1,59.663551,peachy,ameliandjana,None,0,Daily Gut Pulver,0,0,0,0,0
2,99ba56718494f66f,RETURNING,2023-09-02,2000329000,Gut Care Kapseln-NOFLAVOUR,1,59.663551,peachy,ameliandjana,None,0,Gut Care Kapseln,0,0,0,0,0
3,b714a5e095b2bc1b,RETURNING,2023-09-02,2000329008,MCT C8 Pulver-NOFLAVOUR,1,112.152273,dsk,drsimonekoch,None,0,MCT C8 Pulver,0,0,0,0,0
4,b714a5e095b2bc1b,RETURNING,2023-09-02,2000329008,Mood Kapseln-NOFLAVOUR,1,112.152273,dsk,drsimonekoch,None,0,Mood Kapseln,1,0,0,0,0


In [ ]:
import pandas as pd

launch_perf_df = pd.DataFrame([
    ("DAILY FIBER Lemon 330g Doypack DE/FR", "Daily Fiber Drink", "Lemon", "2025-02-24",
     "mass market, health concious consumer", "39,90 €", 15, 39, 0.2777777778, 54, 56),

    ("VEGAN PROTEIN Powder Neutral Doypack 600g DE/EN/FR", "Vegan Protein Pulver", "Neutral", "2025-09-29",
     "health conscious consumer", "29,90 €", 37, 75, 0.3303571429, 112, 134),

    ("PREMIUM GREENS Apple-Kiwi 270g Doypack DE/FR", "Premium Greens", "Apple Kiwi", "2025-02-10",
     "health conscious consumer", "69,90 €", 48, 229, 0.1732851986, 277, 296),

    ("VEGAN GLOW + CLEAR PROTEIN Mango Maracuja 300g Doypack DE Cocreation", "Vegan Glow + Clear Protein Pulver", "Mango Maracuja", "2025-03-10",
     "Looking Good, Wellbeing", "49,90 €", 95, 215, 0.3064516129, 310, 333),

    ("PREMIUM GREENS Mango-Maracuja CoCreation 270g Doypack DE/EN", "Premium Greens", "Mango Maracuja", "2025-02-10",
     "health conscious consumer", "69,90 €", 100, 467, 0.176366843, 567, 609),

    ("RECHARGE DRINK Tropical Fruits 360g Doypack DE/FR", "Recharge Drink", "Tropical Fruits", "2025-07-28",
     "health conscious consumer", "39,90 €", 116, 406, 0.2222222222, 522, 578),

    ("VEGAN PROTEIN Powder Coffee Doypack 600g DE/EN/FR", "Vegan Protein Pulver", "Coffee", "2025-09-29",
     "health conscious consumer", "32,90 €", 142, 244, 0.3678756477, 386, 423),

    ("MAGNESIUM DRINK Lavendel 120g Doypack DE", "Magnesium Drink", "Lavender", "2025-02-24",
     "trendy people, health concious consumer", "32,90 €", 170, 367, 0.3165735568, 537, 552),

    ("DAILY GUT + IMMUNITY Ginger Lemon 240 g Doypack DE/FR Limited Edition", "Daily Gut Pulver", "Ginger Lemon", "2025-10-06",
     "health conscious consumer", "54,90 €", 189, 326, 0.3669902913, 515, 637),

    ("DAILY GUT CREAMY Pistachio 240 g Doypack DE/FR Limited Edition", "Daily Gut Pulver", "Creamy Pistachio", "2025-08-11",
     "mass market, health concious consumer", "49,90 €", 191, 549, 0.2581081081, 740, 774),

    ("SUMMER COLLAGEN Mango Passionfruit 420g Doypack DE/FR Limited Edition", "Summer Collagen", "Mango Passionfruit", "2025-06-24",
     "health conscious consumer", "49,90 €", 427, 820, 0.3424218123, 1247, 1357),

    ("VEGAN PROTEIN Pulver Cookie Dough Doypack 600g DE Cocreation", "Vegan Protein Pulver", "Cookie Dough", "2025-04-22",
     "Looking Good, Wellbeing", "32,90 €", 467, 1042, 0.3094764745, 1509, 1594),

    ("DAILY GUT Powder Raspberry Hibiscus 240 g Doypack DE/FR Limited Edition", "Daily Gut Pulver", "Raspberry Hibiscus", "2025-08-11",
     "mass market, health concious consumer", "49,90 €", 473, 755, 0.3851791531, 1228, 1305),

    ("VEGAN PROTEIN Pulver Chocolate 600g Doypack DE/EN/FR", "Vegan Protein Pulver", "Choco", "2025-01-13",
     "Looking Good, Wellbeing", "32,90 €", 479, 1011, 0.3214765101, 1490, 1638),

    ("DAILY GUT CREAMY Matcha 240 g Doypack DE/FR Limited Edition", "Daily Gut Pulver", "Creamy Matcha", "2025-07-07",
     "mass market, health concious consumer", "49,90 €", 606, 938, 0.3924870466, 1544, 1672),

    ("MAGNESIUM DRINK Blueberry Lemon 120g Doypack DE", "Magnesium Drink", "Blueberry Lemon", "2025-02-24",
     "trendy people, health concious consumer", "32,90 €", 624, 1055, 0.3716497915, 1679, 1787),

    ("VEGAN PROTEIN Pulver Vanilla Cinnamon 600g Doypack DE/EN/FR", "Vegan Protein Pulver", "Vanilla Cinnamon", "2025-01-13",
     "Looking Good, Wellbeing", "32,90 €", 796, 1527, 0.342660353, 2323, 2795),

    ("DAILY GUT + COLLAGEN Powder Creamy Hazelnut 390 g Doypack DE CoCreation", "Daily Gut + Collagen Pulver", "Hazelnut", "2025-05-27",
     "mass market, health concious consumer", "69,90 €", 849, 1735, 0.3285603715, 2584, 2719),

    ("DAILY COLLAGEN Powder 450g Doypack DE/FR", "Daily Collagen Pulver", "Neutral", "2025-04-08",
     "mass market, health concious consumer", "32,90 €", 932, 1342, 0.4098504837, 2274, 2586),

    ("RECHARGE DRINK Lemon 360g Doypack DE/FR", "Recharge Drink", "Lemon", "2025-07-28",
     "health conscious consumer", "39,90 €", 15, 59, 0.1022405773, 74, 175),

    ("HYDRATE DRINK Powder 160g Doypack DE/FR", "Hydrate Drink", "Lemon", "2025-05-19",
     "mass market, health concious consumer", "29,90 €", 8, 34, 0.1707739667, 42, 45),

    ("DAILY GLOW COLLAGEN Raspberry Lemon 135g Doypack DE/FR", "Daily Glow Pulver", "Raspberry Lemon", "2024-12-10",
     "Looking Good, Wellbeing", "34,90 €", 18, 36, 0.3181627509, 54, 55),

], columns=[
    "artikel_name",
    "product",
    "flavour",
    "launch_date",
    "target_group",
    "uvp",
    "nc_amount",
    "rc_amount",
    "nc_share",
    "total_customer",
    "total_quantity"
])

# Format conversions
launch_perf_df["launch_date"] = pd.to_datetime(launch_perf_df["launch_date"])
launch_perf_df["uvp"] = (
    launch_perf_df["uvp"]
    .astype(str)
    .str.replace("€", "")
    .str.replace(",", ".")
    .astype(float)
)

launch_perf_df


,artikel_name,product,flavour,launch_date,target_group,uvp,nc_amount,rc_amount,nc_share,total_customer,total_quantity
0,DAILY FIBER Lemon 330g Doypack DE/FR,Daily Fiber Drink,Lemon,2025-02-24,"mass market, health concious consumer",39.9,15,39,0.277778,54,56
1,VEGAN PROTEIN Powder Neutral Doypack 600g DE/E...,Vegan Protein Pulver,Neutral,2025-09-29,health conscious consumer,29.9,37,75,0.330357,112,134
2,PREMIUM GREENS Apple-Kiwi 270g Doypack DE/FR,Premium Greens,Apple Kiwi,2025-02-10,health conscious consumer,69.9,48,229,0.173285,277,296
3,VEGAN GLOW + CLEAR PROTEIN Mango Maracuja 300g...,Vegan Glow + Clear Protein Pulver,Mango Maracuja,2025-03-10,"Looking Good, Wellbeing",49.9,95,215,0.306452,310,333
4,PREMIUM GREENS Mango-Maracuja CoCreation 270g ...,Premium Greens,Mango Maracuja,2025-02-10,health conscious consumer,69.9,100,467,0.176367,567,609
5,RECHARGE DRINK Tropical Fruits 360g Doypack DE/FR,Recharge Drink,Tropical Fruits,2025-07-28,health conscious consumer,39.9,116,406,0.222222,522,578
6,VEGAN PROTEIN Powder Coffee Doypack 600g DE/EN/FR,Vegan Protein Pulver,Coffee,2025-09-29,health conscious consumer,32.9,142,244,0.367876,386,423
7,MAGNESIUM DRINK Lavendel 120g Doypack DE,Magnesium Drink,Lavender,2025-02-24,"trendy people, health concious consumer",32.9,170,367,0.316574,537,552
8,DAILY GUT + IMMUNITY Ginger Lemon 240 g Doypac...,Daily Gut Pulver,Ginger Lemon,2025-10-06,health conscious consumer,54.9,189,326,0.366990,515,637
9,DAILY GUT CREAMY Pistachio 240 g Doypack DE/FR...,Daily Gut Pulver,Creamy Pistachio,2025-08-11,"mass market, health concious consumer",49.9,191,549,0.258108,740,774


In [ ]:
df = df_expanded.copy()
df["order_date"] = pd.to_datetime(df["order_date"])


In [ ]:
# ============================================================
# DATA-DRIVEN COEFFICIENT CALCULATION
# ============================================================

# -----------------------------------------
# 1) PRICE ELASTICITY (data-driven)
# -----------------------------------------

# Use launch performance: price vs total_quantity
df_price = launch_perf_df[["uvp", "total_quantity"]].dropna()

# Simple log-log regression for elasticity
df_price["log_price"] = np.log(df_price["uvp"])
df_price["log_qty"] = np.log(df_price["total_quantity"])

coef = np.polyfit(df_price["log_price"], df_price["log_qty"], 1)
price_elasticity = coef[0]   # slope = elasticity

print("Estimated price elasticity:", price_elasticity)

# For our launch price = 32 EUR
launch_price = 35.0
avg_price = df_price["uvp"].mean()

# elasticity-based factor
price_elasticity_factor = (launch_price / avg_price) ** price_elasticity
print("Price elasticity factor:", price_elasticity_factor)


# -----------------------------------------
# 2) INFLUENCER / CO-CREATION UPLIFT
# -----------------------------------------

# CoCreation launches
mask_co = launch_perf_df["artikel_name"].str.contains("CoCreation", case=False)

co_avg = launch_perf_df[mask_co]["total_quantity"].mean()
nonco_avg = launch_perf_df[~mask_co]["total_quantity"].mean()

influencer_uplift_factor = co_avg / nonco_avg if nonco_avg > 0 else 1.0
print("Influencer uplift factor:", influencer_uplift_factor)


# -----------------------------------------
# 3) DECEMBER DROP FACTOR (2024 Dec W49-W50)
# -----------------------------------------

df["year"] = df["order_date"].dt.year
df["week"] = df["order_date"].dt.isocalendar().week

# Select CW49-CW50 of 2024
df_dec_2024 = df[(df["year"] == 2024) & (df["week"].isin([49, 50]))]

dec_qty = df_dec_2024["quantity"].sum()

# Baseline: previous 8 weeks (CW41-CW48)
df_baseline_weeks = df[(df["year"] == 2024) & (df["week"].between(41, 48))]
base_qty = df_baseline_weeks["quantity"].sum()

december_drop_factor = dec_qty / base_qty if base_qty > 0 else 1.0
print("December drop factor:", december_drop_factor)


Estimated price elasticity: 0.8671188312249508
Price elasticity factor: 0.8268332703508644
Influencer uplift factor: 1.4021642454788021
December drop factor: 0.1129962855521896


In [ ]:
# ============================================================
# ADD: SALE-PERIOD BASED BEHAVIOUR FEATURES
# ============================================================

sale_df = df.copy()
sale_df["sale_period_name"] = None

# Assign each transaction to its sale period (if any)
for name, start, end in sale_periods:
    mask = (sale_df["order_date"] >= start) & (sale_df["order_date"] <= end)
    sale_df.loc[mask, "sale_period_name"] = name

# 1) last_sale_period_name per customer
last_sale_period = (
    sale_df.dropna(subset=["sale_period_name"])
           .sort_values(["customer_id", "order_date"])
           .groupby("customer_id")
           .tail(1)[["customer_id", "sale_period_name", "order_date"]]
           .rename(columns={"order_date": "last_sale_period_date"})
)

# merge into feature_table later
# ------------------------------------------------------------

# 2) days_since_last_big_sale
def get_last_sale_date(dates):
    if len(dates) == 0:
        return np.nan
    return max(dates)

customer_last_sale = (
    sale_df[sale_df["sale_period_name"].notna()]
    .groupby("customer_id")["order_date"]
    .apply(get_last_sale_date)
    .rename("last_sale_date_any_sale")
)

days_since_last_big_sale = (
    (latest_date - customer_last_sale)
    .dt.days
    .rename("days_since_last_big_sale")
)

# ------------------------------------------------------------

# 3) bought_in_last_sale flag (00 = no, 1 = yes)
# Identify last sale period in dataset
all_sale_periods_sorted = sorted(sale_periods, key=lambda x: x[2])
_, last_sale_start, last_sale_end = all_sale_periods_sorted[-1]

sale_df["bought_in_last_sale"] = (
    (sale_df["order_date"] >= last_sale_start) &
    (sale_df["order_date"] <= last_sale_end)
).astype(int)

bought_last_sale_flag = (
    sale_df.groupby("customer_id")["bought_in_last_sale"]
           .max()
           .rename("bought_in_last_sale")
)


In [ ]:
# ============================================================
# SAFE PRODUCT + FLAVOUR PARSER (supports all formats)
# ============================================================

def parse_product_flavour(x):
    """
    Safely splits product names into product + flavour.
    Handles:
      - 'Daily Gut Pulver–Strawberry'
      - 'Daily Gut Pulver - Strawberry'
      - 'Daily Gut Pulver – Strawberry'
      - NaN or malformed values
    """
    if pd.isna(x):
        return None, None

    x = str(x).strip()

    # Normalize separators
    if "–" in x:
        parts = x.split("–")
    elif "-" in x:
        parts = x.split("-")
    else:
        return x, None

    parts = [p.strip() for p in parts]

    if len(parts) == 1:
        return parts[0], None
    else:
        return parts[0], parts[1]

# Apply parser
df["product_clean"], df["flavour_clean"] = zip(*df["product"].apply(parse_product_flavour))

# Build unique signature
df["product_signature"] = (
    df["product_clean"].astype(str) + "–" + df["flavour_clean"].astype(str)
)


In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# ASSUMPTIONS
# df_expanded → transaction-level dataset (product-flavour level)
# launch_perf_df → launch table with columns: product, flavour, total_quantity, uvp, launch_date
# sale_periods → list of (name, start, end)
# ============================================================

df = df_expanded.copy()
df["order_date"] = pd.to_datetime(df["order_date"])

# ============================================================
# SAFE PRODUCT + FLAVOUR PARSER
# ============================================================

def parse_product_flavour(x):
    """Safely extract product + flavour from any product string."""
    if pd.isna(x):
        return None, None

    x = str(x).strip()

    # Prefer EN DASH first (–)
    if "–" in x:
        parts = x.split("–")
    # Fallback to hyphen
    elif "-" in x:
        parts = x.split("-")
    else:
        return x, None

    parts = [p.strip() for p in parts]

    if len(parts) == 1:
        return parts[0], None
    return parts[0], parts[1]

# APPLY PARSER
df["product_clean"], df["flavour_clean"] = zip(*df["product"].apply(parse_product_flavour))

df["product_signature"] = (
    df["product_clean"].astype(str) + "–" + df["flavour_clean"].astype(str)
)

# ============================================================
# 1) NOVEMBER → DECEMBER DROP FACTOR
# ============================================================

df["month"] = df["order_date"].dt.month
df["year"] = df["order_date"].dt.year

nov_orders = df[(df["year"] == 2024) & (df["month"] == 11)]["order_id"].nunique()
dec_orders = df[(df["year"] == 2024) & (df["month"] == 12)]["order_id"].nunique()

nov_dec_drop_factor = dec_orders / nov_orders if nov_orders > 0 else 1.0
print("1) November → December Drop Factor:", nov_dec_drop_factor)

# ============================================================
# 2) SALE → 14 DAYS POST-LAUNCH PERFORMANCE FACTOR
# ============================================================

sale_periods_sorted = sorted(sale_periods, key=lambda x: pd.to_datetime(x[2]))
last_sale_name, last_sale_start, last_sale_end = sale_periods_sorted[-1]
last_sale_end = pd.to_datetime(last_sale_end)

launch_perf_df = launch_perf_df.copy()
launch_perf_df["days_after_last_sale"] = (
    launch_perf_df["launch_date"] - last_sale_end
).dt.days

post_sale_launches = launch_perf_df[
    (launch_perf_df["days_after_last_sale"] >= 0) &
    (launch_perf_df["days_after_last_sale"] <= 14)
]

if not post_sale_launches.empty:
    post_sale_factor = (
        post_sale_launches["total_quantity"].mean() /
        launch_perf_df["total_quantity"].mean()
    )
else:
    post_sale_factor = 1.0

print("2) Post-Sale (0–14 days) Launch Factor:", post_sale_factor)

# ============================================================
# 3) PRICE BAND SIMILAR LAUNCH PERFORMANCE
# ============================================================

launch_price = 32.90
price_lower = launch_price - 3
price_upper = launch_price + 3

similar_price_products = launch_perf_df[
    launch_perf_df["uvp"].between(price_lower, price_upper)
]

if not similar_price_products.empty:
    price_band_avg_units = similar_price_products["total_quantity"].mean()
else:
    price_band_avg_units = launch_perf_df["total_quantity"].mean()

print("3) Price Band Launch Avg Units:", price_band_avg_units)

# ============================================================
# 4) REAL INFLUENCER UPLIFT (FIT_LAURA)
#    (product + flavour accurate matching)
# ============================================================

df["is_fit_laura"] = (
    df["coupon_influencer"].astype(str).str.lower() == "fit_laura"
).astype(int)

# Create unified signature for launch table
launch_perf_df["product_signature"] = (
    launch_perf_df["product"].astype(str).str.strip() +
    "–" +
    launch_perf_df["flavour"].astype(str).str.strip()
)

# Flag if transaction is a launch purchase
df["is_historical_launch"] = df["product_signature"].isin(
    launch_perf_df["product_signature"].unique()
).astype(int)

fit_customers = df[df["is_fit_laura"] == 1]["customer_id"].unique()
non_fit_customers = df[df["is_fit_laura"] == 0]["customer_id"].unique()

# Fit_Laura purchase rate
if len(fit_customers) > 0:
    launch_purchase_rate_fit = (
        df[(df["customer_id"].isin(fit_customers)) &
           (df["is_historical_launch"] == 1)]["customer_id"].nunique()
        / len(fit_customers)
    )
else:
    launch_purchase_rate_fit = 0

# Non-Fit purchase rate
if len(non_fit_customers) > 0:
    launch_purchase_rate_non_fit = (
        df[(df["customer_id"].isin(non_fit_customers)) &
           (df["is_historical_launch"] == 1)]["customer_id"].nunique()
        / len(non_fit_customers)
    )
else:
    launch_purchase_rate_non_fit = 1

influencer_real_uplift = (
    launch_purchase_rate_fit / launch_purchase_rate_non_fit
    if launch_purchase_rate_non_fit > 0 else 1.0
)

print("4) Influencer Real Uplift (Fit_Laura):", influencer_real_uplift)


1) November → December Drop Factor: 0.8289620328404785
2) Post-Sale (0–14 days) Launch Factor: 1.0
3) Price Band Launch Avg Units: 1160.9
4) Influencer Real Uplift (Fit_Laura): 2.165871484147429


In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================
import pandas as pd
import numpy as np
from datetime import timedelta

!pip install xgboost -q
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


# ============================================================
# 1. INPUTS & BASE PREP
#  - df_expanded: transaction-level data
#  - launch_perf_df: launch performance table (product + flavour)
#  - sale_periods: list of (name, start, end)
#  - price_elasticity_factor, influencer_uplift_factor, december_drop_factor
# ============================================================

df = df_expanded.copy()
df["order_date"] = pd.to_datetime(df["order_date"])
latest_date = df["order_date"].max()
df["customer_status"] = df["customer_status"].astype(str)

# ------------------------------------------------------------
# SAFE PRODUCT + FLAVOUR PARSER
# (handles: "Daily Gut Pulver–Strawberry", "Daily Gut Pulver - Strawberry", etc.)
# ------------------------------------------------------------
def parse_product_flavour(x):
    if pd.isna(x):
        return None, None
    x = str(x).strip()

    # Normalize different possible separators
    if "–" in x:
        parts = x.split("–")
    elif "-" in x:
        parts = x.split("-")
    else:
        # No separator: treat whole string as product, no flavour
        return x, None

    parts = [p.strip() for p in parts]
    if len(parts) == 1:
        return parts[0], None
    else:
        return parts[0], parts[1]

# Apply parser on df_expanded
df["product_clean"], df["flavour_clean"] = zip(*df["product"].apply(parse_product_flavour))

# Build product signature (even if flavour is None it will still become a string)
df["product_signature"] = (
    df["product_clean"].fillna("").astype(str) + "–" +
    df["flavour_clean"].fillna("").astype(str)
)


# ============================================================
# 2. CLEAN LAUNCH PERFORMANCE TABLE (NEW FORMAT)
# ============================================================

launch_perf_df = launch_perf_df.copy()
launch_perf_df["launch_date"] = pd.to_datetime(launch_perf_df["launch_date"])

launch_perf_df["uvp"] = (
    launch_perf_df["uvp"]
    .astype(str)
    .str.replace("€", "")
    .str.replace(",", ".")
    .astype(float)
)

# Build the same signature as in df (product + "–" + flavour)
launch_perf_df["product_signature"] = (
    launch_perf_df["product"].astype(str).str.strip()
    + "–" +
    launch_perf_df["flavour"].astype(str).str.strip()
)

# Mark historical launch purchases correctly in df
df["is_launch_product"] = df["product_signature"].isin(
    launch_perf_df["product_signature"].unique()
).astype(int)


# ============================================================
# 3. SALE PERIOD BASED FEATURES
# ============================================================

sale_df = df.copy()
sale_df["sale_period_name"] = None

# Tag each transaction with sale period name (if inside any sale window)
for name, start, end in sale_periods:
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)
    mask = (sale_df["order_date"] >= start_dt) & (sale_df["order_date"] <= end_dt)
    sale_df.loc[mask, "sale_period_name"] = name

# Last sale date across any sale window (per customer)
customer_last_sale_any = (
    sale_df[sale_df["sale_period_name"].notna()]
    .groupby("customer_id")["order_date"]
    .max()
    .rename("last_sale_date_any_sale")
)

days_since_last_big_sale = (latest_date - customer_last_sale_any).dt.days
days_since_last_big_sale.name = "days_since_last_big_sale"

# Last sale in sale_periods list
sale_periods_sorted = sorted(sale_periods, key=lambda x: pd.to_datetime(x[2]))
_, last_sale_start, last_sale_end = sale_periods_sorted[-1]
last_sale_start = pd.to_datetime(last_sale_start)
last_sale_end = pd.to_datetime(last_sale_end)

sale_df["bought_in_last_sale"] = (
    (sale_df["order_date"] >= last_sale_start) &
    (sale_df["order_date"] <= last_sale_end)
).astype(int)

bought_last_sale_flag = sale_df.groupby("customer_id")["bought_in_last_sale"].max()


# ============================================================
# 4. CUSTOMER FEATURES (RFM + AFFINITIES)
# ============================================================

# Recency
recency = (
    df.groupby("customer_id")["order_date"]
      .max()
      .apply(lambda d: (latest_date - d).days)
      .rename("recency_days")
)

# Frequency
frequency = df.groupby("customer_id")["order_id"].nunique().rename("order_count")

# Total quantity
total_qty = df.groupby("customer_id")["quantity"].sum().rename("total_quantity")

# Affinities
health_aff     = df.groupby("customer_id")["is_health_product"].mean().rename("health_affinity")
hydration_aff  = df.groupby("customer_id")["is_hydration_product"].mean().rename("hydration_affinity")
protein_aff    = df.groupby("customer_id")["is_protein_product"].mean().rename("protein_affinity")

# Previous co-creation product (e.g. Vegan Protein Cookie)
prev_cocreate_aff = df.groupby("customer_id")["is_prev_cocreation_product"].max().rename("prev_cocreation_affinity")

# Influencer (fit_laura) interaction
influencer_aff = df.groupby("customer_id")["is_cocreation_influencer_customer"].max().rename("cocreation_influencer_affinity")

# Sale behaviour basics
sale_freq = df.groupby("customer_id")["is_sale_period"].mean().rename("sale_frequency")
last_sale_date = (
    df[df["is_sale_period"] == 1]
    .groupby("customer_id")["order_date"]
    .max()
)
days_since_sale = (latest_date - last_sale_date).dt.days.rename("days_since_last_sale")

# Sale in last 60 days
df["days_from_latest"] = (latest_date - df["order_date"]).dt.days
df["sale_last_60d_flag"] = ((df["is_sale_period"] == 1) & (df["days_from_latest"] <= 60)).astype(int)
sale_last_60d = df.groupby("customer_id")["sale_last_60d_flag"].max().rename("sale_last_60d")

# Launch affinity
launch_aff = df.groupby("customer_id")["is_launch_product"].mean().rename("launch_affinity")

# Customer status
customer_status = df.groupby("customer_id")["customer_status"].first().rename("customer_status")

# Merge all features
feature_table = pd.concat([
    recency,
    frequency,
    total_qty,
    health_aff,
    hydration_aff,
    protein_aff,
    prev_cocreate_aff,
    influencer_aff,
    sale_freq,
    days_since_sale,
    sale_last_60d,
    launch_aff,
    days_since_last_big_sale,
    bought_last_sale_flag,
    customer_status
], axis=1).reset_index()

# Fill missing values
feature_table["days_since_last_sale"]    = feature_table["days_since_last_sale"].fillna(999)
feature_table["sale_frequency"]          = feature_table["sale_frequency"].fillna(0)
feature_table["sale_last_60d"]           = feature_table["sale_last_60d"].fillna(0)
feature_table["launch_affinity"]         = feature_table["launch_affinity"].fillna(0)
feature_table["days_since_last_big_sale"] = feature_table["days_since_last_big_sale"].fillna(999)
feature_table["bought_in_last_sale"]      = feature_table["bought_in_last_sale"].fillna(0)

# NEW customer flag
feature_table["is_new_customer"] = (
    feature_table["customer_status"].str.upper() == "NEW"
).astype(int)


# ============================================================
# 5. MODEL 2 – XGBOOST PROPENSITY
# ============================================================

feature_table["label_prev_cocreated"] = (
    feature_table["prev_cocreation_affinity"] > 0
).astype(int)

feature_cols = [
    "recency_days",
    "order_count",
    "total_quantity",
    "health_affinity",
    "hydration_affinity",
    "protein_affinity",
    "prev_cocreation_affinity",
    "cocreation_influencer_affinity",
    "sale_frequency",
    "days_since_last_sale",
    "sale_last_60d",
    "launch_affinity",
    "days_since_last_big_sale",
    "bought_in_last_sale",
    "is_new_customer",
]

X = feature_table[feature_cols].fillna(0)
y = feature_table["label_prev_cocreated"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
)

xgb_model.fit(X_train, y_train)
auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])
print("Model 2 AUC:", auc)

feature_table["p_buy_model2"] = xgb_model.predict_proba(X)[:, 1]

model2_total_units = feature_table["p_buy_model2"].sum()
model2_new_units   = feature_table.loc[
    feature_table["is_new_customer"] == 1, "p_buy_model2"
].sum()


# ============================================================
# 6. MODEL 1 – BASELINE (LAUNCH BENCHMARK)
# ============================================================

segment_launches = launch_perf_df[
    launch_perf_df["target_group"].str.contains("health", case=False)
]

mean_launch_units = (
    segment_launches["total_quantity"].mean()
    if not segment_launches.empty
    else launch_perf_df["total_quantity"].mean()
)

model1_units = (
    mean_launch_units *
    price_elasticity_factor *
    influencer_uplift_factor *
    december_drop_factor
)


# ============================================================
# 7. MODEL 3 – SCENARIO (TREND + MULTIPLIERS)
# ============================================================

hydration_df = df[df["is_hydration_product"] == 1].copy()

if hydration_df.empty:
    hydration_momentum = 1.0
else:
    hydration_df["week"] = hydration_df["order_date"].dt.isocalendar().week
    hydration_df["year"] = hydration_df["order_date"].dt.year

    cutoff12 = latest_date - pd.Timedelta(weeks=12)
    cutoff4  = latest_date - pd.Timedelta(weeks=4)

    last12 = hydration_df[hydration_df["order_date"] >= cutoff12]
    last4  = hydration_df[hydration_df["order_date"] >= cutoff4]

    w12 = last12.groupby(["year", "week"])["quantity"].sum().mean()
    w4  = last4.groupby(["year", "week"])["quantity"].sum().mean()

    hydration_momentum = (w4 / w12) if (w12 is not None and w12 > 0) else 1.0

model3_units = (
    model2_total_units *
    hydration_momentum *
    influencer_uplift_factor *
    price_elasticity_factor *
    december_drop_factor
)


# ============================================================
# 8. FINAL ENSEMBLE
# ============================================================

final_units_float = (
    0.50 * model1_units +
    0.30 * model2_total_units +
    0.20 * model3_units
)

final_units_int     = int(round(final_units_float))
final_new_units_int = int(round(model2_new_units))

print("\n=== FINAL FORECAST ===")
print("Model 1:", int(round(model1_units)))
print("Model 2:", int(round(model2_total_units)))
print("Model 3:", int(round(model3_units)))
print("Final blended:", final_units_int)
print("New customers:", final_new_units_int)


# ============================================================
# 9. SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame({
    "Model": [
        "Model 1: Baseline",
        "Model 2: XGBoost",
        "Model 3: Scenario",
        "Final"
    ],
    "Units (rounded)": [
        int(round(model1_units)),
        int(round(model2_total_units)),
        int(round(model3_units)),
        final_units_int
    ]
})

summary_df


Model 2 AUC: 1.0

=== FINAL FORECAST ===
Model 1: 121
Model 2: 5499
Model 3: 789
Final blended: 1868
New customers: 3230


,Model,Units (rounded)
0,Model 1: Baseline,121
1,Model 2: XGBoost,5499
2,Model 3: Scenario,789
3,Final,1868


In [ ]:
import pickle

# Save all variables that app.py needs to run independently
artifacts = {
    "launch_perf_df":          launch_perf_df,
    "feature_table":            feature_table,
    "auc":                      auc,
    "price_elasticity":         price_elasticity,
    "influencer_uplift_factor": influencer_uplift_factor,
    "hydration_momentum":       hydration_momentum,
}

with open("model_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("model_artifacts.pkl saved — run: python app.py")


In [ ]:
!pip install gradio plotly -q

In [ ]:
import gradio as gr
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import time

# ── Month name → index ────────────────────────────────────────────────────────
MONTH_MAP = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

# ── Seasonal index (relative to annual average = 1.0) ─────────────────────────
SEASONAL_INDEX = {
    1: 1.05, 2: 1.02, 3: 1.08, 4: 1.10, 5: 1.12, 6: 1.07,
    7: 0.95, 8: 0.98, 9: 1.03, 10: 1.15, 11: 1.30, 12: 0.75
}

# ── Keyword-based nearest-SKU matcher ────────────────────────────────────────
def find_nearest_skus(characteristics: str, search_terms: str, n: int = 5):
    keywords = (
        characteristics.lower().replace(',', ' ').split() +
        search_terms.lower().split()
    )
    scores = []
    for _, row in launch_perf_df.iterrows():
        haystack = ' '.join([
            str(row.get('artikel_name', '')),
            str(row.get('product', '')),
            str(row.get('flavour', '')),
            str(row.get('target_group', ''))
        ]).lower()
        score = sum(1 for kw in keywords if kw in haystack)
        scores.append((score, row))
    scores.sort(key=lambda x: x[0], reverse=True)
    return [r for _, r in scores[:n]]


# ── Core forecast function ────────────────────────────────────────────────────
def generate_forecast(product_name, characteristics, search_terms,
                      unit_price, month_name, collab_mode):
    logs = []
    ts = lambda: time.strftime('%H:%M:%S')

    logs.append(f"[{ts()}] [USER]    EXECUTING FORECAST FOR {product_name.upper()}...")
    logs.append(f"[{ts()}] [AI]      ANALYZING CHARACTERISTICS: {characteristics.upper()}...")

    # --- Nearest-SKU matching ---
    nearest_skus = find_nearest_skus(characteristics, search_terms, n=5)
    if nearest_skus:
        best = nearest_skus[0]
        logs.append(f"[{ts()}] [MATCH]   NEAREST HISTORICAL SKU: {best.get('product','N/A')} {best.get('flavour','')}")
    else:
        nearest_skus = launch_perf_df.sort_values('total_quantity', ascending=False).head(5).to_dict('records')
        logs.append(f"[{ts()}] [MATCH]   USING TOP VOLUME SKUs AS REFERENCE")

    logs.append(f"[{ts()}] [COMPUTE] PROCESSING ENSEMBLE MODELS...")

    # --- Dynamic factors ---
    seasonal_factor  = SEASONAL_INDEX.get(MONTH_MAP.get(month_name, 11), 1.0)
    avg_price        = launch_perf_df['uvp'].mean()
    dyn_price_factor = (unit_price / avg_price) ** price_elasticity if avg_price > 0 else 1.0
    dyn_inf_factor   = influencer_uplift_factor if collab_mode == 'Co-Creation' else 1.0

    # --- Model 1: Baseline (average of nearest SKUs x factors) ---
    ref_qty = np.mean([r['total_quantity'] for r in nearest_skus])
    m1 = ref_qty * dyn_price_factor * dyn_inf_factor * seasonal_factor

    # --- Model 2: XGBoost propensity sum (re-scaled) ---
    scale    = dyn_price_factor * dyn_inf_factor * seasonal_factor
    m2_total = feature_table['p_buy_model2'].sum() * scale
    m2_new   = feature_table.loc[feature_table['is_new_customer'] == 1, 'p_buy_model2'].sum() * scale

    # --- Model 3: Scenario (hydration momentum x factors) ---
    m3 = m2_total * hydration_momentum * dyn_inf_factor * dyn_price_factor * seasonal_factor

    # --- Blended ensemble ---
    blended  = int(round(0.50 * m1 + 0.30 * m2_total + 0.20 * m3))
    new_cust = int(round(m2_new))

    logs.append(f"[{ts()}] [MODEL1]  BASELINE  -> {int(round(m1))} units")
    logs.append(f"[{ts()}] [MODEL2]  XGBOOST   -> {int(round(m2_total))} units  (AUC: {auc:.2f})")
    logs.append(f"[{ts()}] [MODEL3]  SCENARIO  -> {int(round(m3))} units")
    logs.append(f"[{ts()}] [FINAL]   BLENDED   -> {blended} units | NEW CUSTOMERS: {new_cust}")

    # --- Ensemble bar chart ---
    names  = ['BASELINE', 'XGBOOST', 'SCENARIO', 'BLENDED']
    values = [int(round(m1)), int(round(m2_total)), int(round(m3)), blended]
    confidence = round(min(0.99, 0.70 + 0.06 * len([s for s in nearest_skus if s])), 2)

    fig = go.Figure()
    fig.add_trace(go.Bar(x=names[:3], y=values[:3],
                         marker_color='#b0bec5', name='Base Models'))
    fig.add_trace(go.Bar(x=[names[3]], y=[values[3]],
                         marker_color='#1565C0', name='Weighted Result'))
    fig.update_layout(
        title=dict(
            text=f'Ensemble Comparison Analysis  |  CONFIDENCE SCORE: {confidence}',
            font=dict(size=13)
        ),
        plot_bgcolor='white', paper_bgcolor='white',
        bargap=0.35, height=380,
        yaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
        legend=dict(orientation='h', yanchor='bottom', y=-0.28, xanchor='center', x=0.5),
        margin=dict(l=30, r=30, t=70, b=40)
    )

    # --- Reference SKU table ---
    avg_ref  = launch_perf_df['total_quantity'].mean()
    ref_rows = []
    for i, row in enumerate(nearest_skus, start=1):
        pct = ((row['total_quantity'] / avg_ref) - 1) * 100 if avg_ref > 0 else 0
        ref_rows.append({
            'SKU ID':    f'S-{i:03d} Reference',
            'Product':   f"{row.get('product','N/A')} - {row.get('flavour','')}",
            'Base Adj.': f"{'+' if pct >= 0 else ''}{pct:.0f}%"
        })
    ref_df = pd.DataFrame(ref_rows)

    return blended, new_cust, fig, '\n'.join(logs), ref_df


# ── Gradio layout ─────────────────────────────────────────────────────────────
CSS = """
body, .gradio-container { font-family: 'Inter', sans-serif !important;
                           background: #f4f6f9 !important; }
.section-label { font-size:.68rem; font-weight:700; color:#888;
                 letter-spacing:.10em; text-transform:uppercase; margin-bottom:6px; }
#telemetry-box textarea { background:#1a1a2e !important; color:#7ec8e3 !important;
                           font-family:monospace !important; font-size:.78rem !important; }
"""

with gr.Blocks(css=CSS, title='Demand Forecasting Ensemble') as demo:

    # Header
    gr.HTML("""
    <div style='background:#fff;border-bottom:1px solid #e0e0e0;
                padding:14px 24px;display:flex;align-items:center;
                justify-content:space-between;border-radius:8px 8px 0 0;'>
      <div>
        <div style='font-size:1.15rem;font-weight:700;color:#111;'>
          📦 Demand Forecasting Ensemble
        </div>
        <div style='font-size:.7rem;color:#aaa;letter-spacing:.06em;'>
          MASTER THESIS RESEARCH PROTOTYPE V1.2
        </div>
      </div>
      <div style='display:flex;gap:28px;align-items:center;'>
        <span style='color:#1565C0;font-size:.82rem;font-weight:600;'>
          &#9889; Model Status: <b>Active</b>
        </span>
        <span style='color:#aaa;font-size:.75rem;'>REF: THESIS_2024_0422</span>
      </div>
    </div>
    """)

    with gr.Row(equal_height=False):

        # Left: input parameters
        with gr.Column(scale=1, min_width=250):
            gr.HTML("<div class='section-label'>&#9881; Input Parameters</div>")
            inp_product = gr.Textbox(label='PRODUCT NAME',
                                     value='NEW PERFORMANCE SNACK')
            inp_chars   = gr.Textbox(label='CHARACTERISTICS',
                                     value='HIGH PROTEIN, VEGAN, COCONUT')
            inp_terms   = gr.Textbox(label='KEY SEARCH TERMS',
                                     value='protein vegan cookie')
            with gr.Row():
                inp_price = gr.Number(label='UNIT PRICE (EUR)', value=32.9, minimum=0.1)
                inp_month = gr.Dropdown(label='MONTH INDEX',
                                        choices=list(MONTH_MAP.keys()), value='Nov')
            inp_collab = gr.Radio(label='COLLABORATION MODE',
                                  choices=['Co-Creation', 'Standard'],
                                  value='Co-Creation')
            btn = gr.Button('Generate Forecast >', variant='primary', size='lg')

        # Center: metrics + chart
        with gr.Column(scale=2):
            with gr.Row():
                with gr.Column():
                    gr.HTML("<div class='section-label'>&#128230; Estimate. Blended Volume</div>")
                    out_blended  = gr.Number(label='', value=0, interactive=False)
                with gr.Column():
                    gr.HTML("<div class='section-label'>&#128100; New Customer Proj.</div>")
                    out_new_cust = gr.Number(label='', value=0, interactive=False)
            out_chart = gr.Plot(label='')

        # Right: telemetry + reference set
        with gr.Column(scale=1, min_width=270):
            gr.HTML("<div class='section-label'>&#9000; Process Telemetry</div>")
            out_log = gr.Textbox(
                label='', lines=9, max_lines=12, interactive=False,
                placeholder='Waiting for forecast run...',
                elem_id='telemetry-box'
            )
            gr.HTML("<div class='section-label' style='margin-top:14px;'>&#128203; Model Reference Set</div>")
            out_ref = gr.DataFrame(
                label='', headers=['SKU ID', 'Product', 'Base Adj.'],
                interactive=False, wrap=True
            )

    # Footer
    gr.HTML("""
    <div style='text-align:center;font-size:.68rem;color:#bbb;padding:10px 0 4px;'>
      &copy; 2024 Predictive Analytics Laboratory &nbsp;|&nbsp;
      <span style='color:#4caf50;'>&#9679; Kernel Online</span> &nbsp;|&nbsp;
      Build: 4.2.0-STABLE
    </div>
    """)

    btn.click(
        fn=generate_forecast,
        inputs=[inp_product, inp_chars, inp_terms,
                inp_price, inp_month, inp_collab],
        outputs=[out_blended, out_new_cust, out_chart, out_log, out_ref]
    )

demo.launch(share=True, debug=False)
